# Phase 3 · S3 — Engagement-weighted (F3) edgewise training, 2×2 (họ model × weighting)

**Đóng góp chính của dự án.** Giả thuyết: tín hiệu **review-length** (F3 = `(rating/5)·log1p(review_token_count)`)
là engagement-weight tốt và **transfer xuyên 2 họ model**:
- **chainRec**: weight `loss_pos` bằng F3 qua hook `w_pos` trong `edgewise_loss`.
- **ALS**: dùng F3 làm **confidence** (giá trị ô CSR), giống Phase 2 nhưng trong index-space chainRec.

So bảng **2×2** trên *cùng* `data_test`/pool/mask, dùng đúng `rank_eval` của S1a/S2:

|            | vanilla            | +F3            |
|------------|--------------------|----------------|
| chainRec   | (lấy từ S2)        | train ở mục 4  |
| ALS        | (lấy từ S2)        | train ở mục 5  |

**Nền tảng:** Kaggle GPU T4 + Internet On (load reviews parquet & bridge từ HF `vngclinh/goodreads-preprocessed`).
Vanilla baselines lấy lại từ `s2/s2_headtohead.json` để so cùng trainer/seed (không train lại).


## 0 · Setup

In [ ]:
import os, json, time, pickle, random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Literal
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import implicit
except ImportError:
    os.system("pip install -q implicit"); import implicit
from scipy.sparse import csr_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| implicit", implicit.__version__)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

## 1 · Config + load artifacts (chainRec processed + bridge + S2 baselines)

In [ ]:
HF_REPO    = "vngclinh/goodreads-preprocessed"
PROC_LOCAL = Path("/kaggle/working/processed")
CKPT_LOCAL = Path("/kaggle/working/chainrec")
S2_LOCAL   = Path("/kaggle/working/s2")
S3_LOCAL   = Path("/kaggle/working/s3"); S3_LOCAL.mkdir(parents=True, exist_ok=True)

EVAL_USERS  = 5000
BATCH_USERS = 64
K_LIST      = (10, 20)

from huggingface_hub import hf_hub_download, HfApi
def _hf(rel): return hf_hub_download(HF_REPO, rel, repo_type="dataset", token=HF_TOKEN)
def load_npy(n):
    p = PROC_LOCAL/n; return np.load(p if p.exists() else _hf(f"chainrec/processed/{n}"))
def load_pkl_proc(n):
    p = PROC_LOCAL/n; return pickle.load(open(p if p.exists() else _hf(f"chainrec/processed/{n}"), "rb"))
def load_meta():
    p = PROC_LOCAL/"meta.json"
    return json.loads(Path(p if p.exists() else _hf("chainrec/processed/meta.json")).read_text())

data_train    = load_npy("data_train.npy")
data_test     = load_npy("data_test.npy")
user_item_map = load_pkl_proc("user_item_map.pkl")
meta          = load_meta()
N_ITEM, N_USER, N_STAGE = meta["n_item"], meta["n_user"], meta["n_stage"]
REC = N_STAGE - 1

# bridge int<->string (từ S2) + baselines S2
bridge_p = S2_LOCAL/"id_bridge.pkl"
bridge   = pickle.load(open(bridge_p if bridge_p.exists() else _hf("s2/id_bridge.pkl"), "rb"))
idx2user_str = np.asarray(bridge["idx2user_str"], dtype=object)
idx2book_str = np.asarray(bridge["idx2book_str"], dtype=object)
s2_base = json.loads(Path(_hf("s2/s2_headtohead.json")).read_text())

print(f"n_user={N_USER:,}  n_item={N_ITEM:,}  REC={REC}  train_edges={len(data_train):,}")
print("S2 baselines:", {k: round(v["AUC"],4) for k,v in s2_base.items()})

## 2 · Dựng trọng số **F3** cho từng training edge

`F3 = (rating/5)·log1p(review_token_count)`, join qua bridge → (user_str, book_str) → reviews parquet.
Coverage *một phần* (chỉ edge có review). Thiếu → fill = trung bình F3 (trung tính), rồi chuẩn hoá **mean=1**
để giữ scale loss giống vanilla. Bản raw (thiếu→1.0) dùng cho confidence ALS.

In [ ]:
import pandas as pd
api = HfApi()
parq = sorted(f for f in api.list_repo_files(HF_REPO, repo_type="dataset")
              if f.startswith("data/") and f.endswith(".parquet"))
print(f"{len(parq)} parquet files:", [Path(p).stem for p in parq])

user_keep = set(x for x in idx2user_str.tolist() if x is not None)
print(f"users in 48K sample: {len(user_keep):,}")

parts = []
for f in parq:
    d = pd.read_parquet(_hf(f), columns=["user_id","book_id","rating","review_token_count"])
    d["user_id"] = d["user_id"].astype(str); d["book_id"] = d["book_id"].astype(str)
    d = d[d["user_id"].isin(user_keep)]
    parts.append(d); print(f"   {Path(f).stem:42} kept {len(d):,}")
rev = pd.concat(parts, ignore_index=True)
rev["f3"] = (rev["rating"].astype(float)/5.0) * np.log1p(rev["review_token_count"].astype(float))
rev = rev.groupby(["user_id","book_id"], as_index=False)["f3"].max()   # dedup theo cặp
print(f"review pairs (joined to 48K users): {len(rev):,}")

# map mọi edge data_train -> (user_str, book_str) -> f3
us = idx2user_str[data_train[:,0]].astype(str)
bs = idx2book_str[data_train[:,1]].astype(str)
edges = pd.DataFrame({"user_id": us, "book_id": bs})
edges = edges.merge(rev, on=["user_id","book_id"], how="left")
f3_raw = edges["f3"].values.astype(np.float64)               # NaN nơi không có review
cov = np.isfinite(f3_raw).mean()
fill = float(np.nanmean(f3_raw))
f3_filled = np.where(np.isfinite(f3_raw), f3_raw, fill)
w_train = (f3_filled / f3_filled.mean()).astype(np.float32)  # mean=1 cho edgewise loss
print(f"F3 coverage trên train edges: {cov*100:.1f}%  | fill(mean F3)={fill:.3f}  | "
      f"w_train mean={w_train.mean():.3f} std={w_train.std():.3f}")

# weight cho ALS confidence (raw F3, thiếu->1.0) trên recommend-edge
rec_mask = data_train[:,2] == REC
w_rec_raw = np.where(np.isfinite(f3_raw[rec_mask]), f3_raw[rec_mask], 1.0).astype(np.float32)
print(f"recommend edges: {rec_mask.sum():,}  | F3 coverage trên rec-edge: "
      f"{np.isfinite(f3_raw[rec_mask]).mean()*100:.1f}%")

## 3 · chainRec model + `rank_eval` + scorers (giống S1a/S2)

In [ ]:
@dataclass
class ModelConfig:
    n_user: int; n_item: int
    n_stage: int = 4; embed_dim: int = 16
    beta: float = 1.0; learn_beta: bool = True
    l2: float = 0.01; lr: float = 0.001
    batch_size: int = 2048; n_neg: int = 1
    n_epochs: int = 30; patience: int = 5
    sampler: Literal["uniform","stagewise"] = "uniform"
    device: str = DEVICE

class ChainRecModel(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg
        K, L = cfg.embed_dim, cfg.n_stage
        self.user_emb=nn.Embedding(cfg.n_user,K); self.item_emb=nn.Embedding(cfg.n_item,K)
        self.stage_emb=nn.Embedding(L,K); self.b0=nn.Parameter(torch.zeros(1))
        self.b_user=nn.Embedding(cfg.n_user,1); self.b_item=nn.Embedding(cfg.n_item,1)
        lb=torch.log(torch.tensor(float(cfg.beta)))
        if cfg.learn_beta: self.log_beta=nn.Parameter(lb)
        else: self.register_buffer("log_beta",lb)
        for e in [self.user_emb,self.item_emb,self.stage_emb]: nn.init.xavier_uniform_(e.weight)
        for b in [self.b_user,self.b_item]: nn.init.zeros_(b.weight)
    @property
    def beta(self): return torch.clamp(self.log_beta.exp(), min=1.0)
    def _intention(self,u,i,l): return (self.stage_emb(l)*self.item_emb(i)*self.user_emb(u)).sum(-1)
    def _rect(self,d): b=self.beta; return F.softplus(b*d)/b
    def score(self,u,i,target_stage):
        B=u.shape[0]; bias=self.b0+self.b_user(u).squeeze(-1)+self.b_item(i).squeeze(-1)
        acc=torch.zeros(B,device=u.device)
        for lp in range(target_stage,self.cfg.n_stage):
            l_t=torch.full((B,),lp,dtype=torch.long,device=u.device)
            acc=acc+self._rect(self._intention(u,i,l_t))
        return bias+acc
    def edgewise_terms(self,u,i,l_star):
        L=self.cfg.n_stage; bias=self.b0+self.b_user(u).squeeze(-1)+self.b_item(i).squeeze(-1)
        dp=torch.stack([self._rect(self._intention(u,i,torch.full((u.shape[0],),l,dtype=torch.long,device=u.device)))
                        for l in range(L)],dim=1)
        suffix=dp.flip(dims=[1]).cumsum(dim=1).flip(dims=[1]); s=bias.unsqueeze(1)+suffix
        lc=l_star.clamp(0,L-1)
        s_l=s.gather(1,lc.unsqueeze(1)).squeeze(1)
        s_n=s.gather(1,(l_star+1).clamp(0,L-1).unsqueeze(1)).squeeze(1)
        s_n=torch.where(l_star==L-1,torch.full_like(s_n,-1e9),s_n)
        p_l=torch.sigmoid(s_l); p_n=torch.sigmoid(s_n)
        p_cap=(1.0-torch.exp(-self._rect(self._intention(u,i,lc)))).clamp(min=1e-8)
        return p_l,p_n,p_cap

def build_model(sampler="uniform"):
    cfg=ModelConfig(n_user=N_USER,n_item=N_ITEM,n_stage=N_STAGE,sampler=sampler)
    return ChainRecModel(cfg).to(DEVICE), cfg

@torch.no_grad()
def make_chainrec_scorer(model, target_stage):
    model.eval()
    item_emb=model.item_emb.weight; b_item=model.b_item.weight.squeeze(-1); stage_w=model.stage_emb.weight
    def fn(u):
        uvec=model.user_emb(u); bias=(model.b0+model.b_user(u).squeeze(-1)).unsqueeze(1)
        acc=torch.zeros(u.shape[0],item_emb.shape[0],device=u.device)
        for l in range(target_stage,model.cfg.n_stage):
            acc=acc+model._rect((uvec*stage_w[l].unsqueeze(0))@item_emb.t())
        return bias+b_item.unsqueeze(0)+acc
    return fn

@torch.no_grad()
def make_als_scorer(U,V,device="cuda"):
    Ut=torch.as_tensor(U,dtype=torch.float32,device=device); Vt=torch.as_tensor(V,dtype=torch.float32,device=device)
    def fn(u): return Ut[u]@Vt.t()
    return fn

@torch.no_grad()
def rank_eval(score_fn, test_pairs, user_item_map, n_item, pos_stage=None,
              K_list=(10,20), batch_users=64, n_eval_users=None, device="cuda", seed=999):
    pairs = test_pairs if pos_stage is None else test_pairs[test_pairs[:,2]==pos_stage]
    if n_eval_users is not None and len(pairs)>n_eval_users:
        rng=np.random.default_rng(seed); pairs=pairs[rng.choice(len(pairs),size=n_eval_users,replace=False)]
    aucs=[]; hits={k:[] for k in K_list}; ndcg={k:[] for k in K_list}
    for st in range(0,len(pairs),batch_users):
        chunk=pairs[st:st+batch_users]
        u=torch.tensor(chunk[:,0],dtype=torch.long,device=device); pos=torch.tensor(chunk[:,1],dtype=torch.long,device=device)
        scores=score_fn(u); B=u.shape[0]; ar=torch.arange(B,device=device)
        pos_score=scores[ar,pos].clone(); seen_cnt=torch.zeros(B,device=device)
        for b in range(B):
            seen=user_item_map.get(int(u[b]),())
            if seen:
                idx=torch.tensor(list(seen),dtype=torch.long,device=device); scores[b,idx]=float("-inf"); seen_cnt[b]=len(seen)
        scores[ar,pos]=pos_score
        rank=(scores>pos_score.unsqueeze(1)).sum(1).float(); neg=(n_item-seen_cnt).clamp(min=1)
        aucs.append((1.0-rank/neg).cpu().numpy()); rnp=rank.cpu().numpy()
        for k in K_list:
            hit=rnp<k; hits[k].append(hit.astype(float)); ndcg[k].append(np.where(hit,1.0/np.log2(rnp+2),0.0))
        del scores
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    res={"AUC":float(np.concatenate(aucs).mean()),"n_eval":int(len(pairs))}
    for k in K_list:
        res[f"Recall@{k}"]=float(np.concatenate(hits[k]).mean()); res[f"NDCG@{k}"]=float(np.concatenate(ndcg[k]).mean())
    return res

## 4 · Train **chainRec + F3** (stagewise, early-stop R@10, `w_pos`=F3)

Cùng trainer/seed như S1a vanilla → chỉ khác hệ số `w_pos`. Vanilla lấy từ S2 (`chainRec(stagewise)`).

In [ ]:
def edgewise_loss(model, up,ip,lp, un,ing,ln, l2, w_pos=None):
    p_pos,_,_   = model.edgewise_terms(up,ip,lp)
    _,p_n,p_cap = model.edgewise_terms(un,ing,ln)
    lpos = torch.log(p_pos.clamp(min=1e-8))
    loss_pos = -(w_pos*lpos).mean() if w_pos is not None else -lpos.mean()
    loss_neg = -(torch.log((1-p_n).clamp(min=1e-8)) + torch.log(p_cap)).mean()
    l2_loss  = l2*(model.user_emb.weight.norm(2)**2 + model.item_emb.weight.norm(2)**2)/(model.cfg.n_user+model.cfg.n_item)
    return loss_pos + loss_neg + l2_loss

class ChainDatasetW(Dataset):
    def __init__(self, data, w, n_item):
        self.data=data; self.w=w; self.n_item=n_item; self.rng=np.random.default_rng(SEED)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        u,i,l=[int(x) for x in self.data[idx]]; ni=int(self.rng.integers(0,self.n_item))
        return (u,i,l,u,ni,l, np.float32(self.w[idx]))

@torch.no_grad()
def sampled_recall(model, data_test, uim, n_item, target_stage, n_neg=500, k=10, device="cuda"):
    model.eval(); rng=np.random.default_rng(999); hits=[]
    for u,ipos,l in data_test:
        u,ipos=int(u),int(ipos)
        if int(l)!=target_stage: continue
        pos=uim.get(u,set()); negs=[]; t=0
        while len(negs)<n_neg and t<n_neg*5:
            c=rng.integers(0,n_item)
            if c not in pos and c!=ipos: negs.append(c)
            t+=1
        items=np.array([ipos]+negs)
        sc=model.score(torch.full((len(items),),u,dtype=torch.long,device=device),
                       torch.tensor(items,dtype=torch.long,device=device), target_stage).cpu().numpy()
        hits.append(int(int(np.where(np.argsort(-sc)==0)[0][0])<k))
    return float(np.mean(hits))

def train_f3(sampler, data_train, w_train, n_epochs=30, patience=5, tag="f3"):
    model, cfg = build_model(sampler); opt=torch.optim.Adam(model.parameters(), lr=cfg.lr)
    dl=DataLoader(ChainDatasetW(data_train, w_train, N_ITEM), batch_size=cfg.batch_size,
                  shuffle=True, num_workers=4, pin_memory=True)
    best,pc,hist=-1.0,0,[]; save=str(S3_LOCAL/f"chainrec_{sampler}_{tag}.pt")
    print(f"{'Ep':>3} | {'train':>8} | {'R@10(s)':>8} | {'s':>5}")
    for ep in range(1,n_epochs+1):
        model.train(); t0=time.time(); tot=nb=0
        for u,i,l,un,ni,ln,w in dl:
            u,i,l=u.to(DEVICE),i.to(DEVICE),l.to(DEVICE); un,ni,ln=un.to(DEVICE),ni.to(DEVICE),ln.to(DEVICE); w=w.to(DEVICE)
            opt.zero_grad(); loss=edgewise_loss(model,u,i,l,un,ni,ln,cfg.l2,w_pos=w)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); tot+=loss.item(); nb+=1
        r10=sampled_recall(model,data_test,user_item_map,N_ITEM,REC,device=DEVICE)
        print(f"{ep:>3} | {tot/nb:>8.4f} | {r10:>8.4f} | {time.time()-t0:>4.1f}")
        hist.append({"epoch":ep,"train_loss":tot/nb,"recall@10_sampled":r10})
        if r10>best: best,pc=r10,0; torch.save(model.state_dict(),save)
        else:
            pc+=1
            if pc>=patience: print(f"Early stop @ {ep} (best R@10={best:.4f})"); break
    model.load_state_dict(torch.load(save,map_location=DEVICE))
    json.dump(hist, open(S3_LOCAL/f"history_{sampler}_{tag}.json","w"), indent=2)
    return model

cr_f3 = train_f3("stagewise", data_train, w_train)
cr_f3_res = rank_eval(make_chainrec_scorer(cr_f3, REC), data_test, user_item_map, N_ITEM,
                      pos_stage=REC, K_list=K_LIST, batch_users=BATCH_USERS, n_eval_users=EVAL_USERS, device=DEVICE)
print("chainRec+F3:", {k:round(v,4) for k,v in cr_f3_res.items()})

## 5 · Train **ALS + F3** (confidence = F3 raw) trong index-space chainRec

In [ ]:
rows = data_train[rec_mask,0].astype(np.int32)
cols = data_train[rec_mask,1].astype(np.int32)
ui_f3 = csr_matrix((w_rec_raw, (rows, cols)), shape=(N_USER, N_ITEM))
print(f"ALS+F3 matrix nnz={ui_f3.nnz:,}  conf mean={w_rec_raw.mean():.3f}")
try:
    als = implicit.als.AlternatingLeastSquares(factors=64, iterations=20, regularization=0.1,
                                               alpha=40, random_state=SEED, use_gpu=False)
except TypeError:
    als = implicit.als.AlternatingLeastSquares(factors=64, iterations=20, regularization=0.1,
                                               random_state=SEED, use_gpu=False); ui_f3 = ui_f3*40.0
t0=time.time(); als.fit(ui_f3); print(f"ALS+F3 fit {time.time()-t0:.1f}s")
Uf,Vf = np.asarray(als.user_factors,dtype=np.float32), np.asarray(als.item_factors,dtype=np.float32)
np.save(S3_LOCAL/"als_f3_user_factors.npy", Uf)
als_f3_res = rank_eval(make_als_scorer(Uf,Vf,device=DEVICE), data_test, user_item_map, N_ITEM,
                       pos_stage=REC, K_list=K_LIST, batch_users=BATCH_USERS, n_eval_users=EVAL_USERS, device=DEVICE)
print("ALS+F3:", {k:round(v,4) for k,v in als_f3_res.items()})

## 6 · Bảng 2×2 (họ model × weighting) + lưu

In [ ]:
grid = {
    "chainRec | vanilla": s2_base["chainRec(stagewise)"],
    "chainRec | +F3":     cr_f3_res,
    "ALS      | vanilla": s2_base["ALS(vanilla)"],
    "ALS      | +F3":     als_f3_res,
    "itemPop  | (floor)": s2_base["itemPop"],
}
print(f"{'config':22}{'AUC':>9}{'R@10':>9}{'N@10':>9}{'R@20':>9}")
for name,r in grid.items():
    print(f"{name:22}{r['AUC']:>9.4f}{r['Recall@10']:>9.4f}{r['NDCG@10']:>9.4f}{r['Recall@20']:>9.4f}")

def delta(a,b,m): return f"{(grid[a][m]-grid[b][m]):+.4f} ({(grid[a][m]/grid[b][m]-1)*100:+.1f}%)"
print("\n=== Δ +F3 vs vanilla (AUC | R@10) ===")
print("chainRec:", delta("chainRec | +F3","chainRec | vanilla","AUC"), "|", delta("chainRec | +F3","chainRec | vanilla","Recall@10"))
print("ALS     :", delta("ALS      | +F3","ALS      | vanilla","AUC"), "|", delta("ALS      | +F3","ALS      | vanilla","Recall@10"))
json.dump(grid, open(S3_LOCAL/"s3_ablation_2x2.json","w"), indent=2)
print("\nSaved s3_ablation_2x2.json")

## 7 · Push + bước tiếp theo (S4)

In [ ]:
to_push = sorted(S3_LOCAL.glob("*"))
print("HF_TOKEN set?", HF_TOKEN is not None, "| files:", [p.name for p in to_push])
if HF_TOKEN and to_push:
    api = HfApi()
    for p in to_push:
        if p.stat().st_size > 200e6: print("  skip (lớn):", p.name); continue
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f"s3/{p.name}",
                        repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
        print("  ✓", p.name)
    print(f"Done → {HF_REPO}/s3/")
else:
    print("Không push (thiếu token). Artifact ở /kaggle/working/s3.")

## 8 · Đọc kết quả & S4

**Cách đọc bảng 2×2 (mục 6):**
- Δ(+F3 − vanilla) trong **mỗi họ** > 0 ⇒ review-length là engagement-weight có ích & **transfer xuyên 2 họ**
  (đây là luận điểm chính, kể cả khi chainRec vẫn thua ALS về *mức tuyệt đối*).
- F3 chỉ phủ một phần edge (in coverage ở mục 2) → nếu Δ nhỏ/phẳng, một phần do coverage thấp; vẫn là finding
  trung thực ("engagement-weighting chỉ tác động nơi quan sát được review").
- itemPop là sàn; AUC<0.5 do positive là sách niche đã bị mask.

**S4 (mở rộng ablation, để chốt báo cáo):**
1. Bọc mục 4–5 trong vòng **≥3 seed** (đổi `random_state`/`torch.manual_seed`), báo mean±std.
2. Thêm biến thể **+votes** (`edge_weight = (rating/5)·log1p(n_votes)`) để tái xác nhận votes gây popularity bias (âm).
3. (Tùy chọn) thêm cột **hybrid ALS+chainRec** (rank-fusion) nếu muốn.
4. Tổng hợp 1 bảng cuối + đồng bộ README/báo cáo; xử lý `has_recsys/` (ngoài báo cáo).
